In [9]:
!pip install -q transformers accelerate datasets evaluate scikit-learn pandas numpy


In [10]:
import os
import torch
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

drive.mount('/content/drive')
BASE_DIR = "/content/drive/My Drive/afroxlmr_detector"
os.makedirs(BASE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
AFROXLMR_MODEL = "Davlan/afro-xlmr-base"


LABEL2ID = {"human": 0, "machine": 1}
ID2LABEL = {0: "human", 1: "machine"}
NUM_LABELS = 2

MAX_LENGTH   = 256
BATCH_SIZE   = 8
EPOCHS       = 5
LEARNING_RATE = 2e-5
SEED         = 42
PHASE1_METRICS_JSON = "/content/drive/MyDrive/afroxlmr_detector/baseline_metrics.json"

In [12]:
df = pd.read_csv("/content/drive/MyDrive/afroxlmr_detector/merged_dataset (1).csv")


df.columns = df.columns.str.strip().str.lower()


df = df.rename(columns={
    "text_generated":   "text",
    "language_code":    "language",
    "model_identifier": "model"
})

if df["label"].dtype == object:
    df["label"] = df["label"].str.strip().str.lower().map({"human": 0, "machine": 1})
df["label"] = df["label"].astype(int)


df["language"] = df["language"].str.strip().str.lower()

In [13]:
assert set(df["label"].unique()).issubset({0, 1}), \
    "label column must contain only 0/1 or 'human'/'machine'"
assert {"zu", "xh", "ss"}.issubset(set(df["language"].unique())), \
    "language column must contain 'zu', 'xh', and 'ss' values"

In [14]:

train_val_df = df[df["language"].isin(["zu", "xh"])].copy()
siswati_df   = df[df["language"] == "ss"].copy()


train_df, val_df = train_test_split(
    train_val_df, test_size=0.2, random_state=SEED, stratify=train_val_df["label"]
)

In [15]:

print(f"✓ Loaded {len(df)} total samples from CSV\n")
print(f"Language breakdown:")
print(df["language"].value_counts().to_string())

print(f"\nSplit summary:")
print(f"  Train (Zulu+Xhosa) : {len(train_df):>5} samples")
print(f"  Val   (Zulu+Xhosa) : {len(val_df):>5} samples")
print(f"  Test  (Siswati)    : {len(siswati_df):>5} samples")

print(f"\nTrain label distribution:")
print(train_df["label"].map({0:"human",1:"machine"}).value_counts().to_string())
print(f"\nSiswati label distribution:")
print(siswati_df["label"].map({0:"human",1:"machine"}).value_counts().to_string())


✓ Loaded 953 total samples from CSV

Language breakdown:
language
xh    341
zu    319
ss    293

Split summary:
  Train (Zulu+Xhosa) :   528 samples
  Val   (Zulu+Xhosa) :   132 samples
  Test  (Siswati)    :   293 samples

Train label distribution:
label
human      289
machine    239

Siswati label distribution:
label
human      183
machine    110


In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(AFROXLMR_MODEL)
print(f"\n✓ Loaded tokenizer: {AFROXLMR_MODEL}")

from datasets import Dataset

def tokenize(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

def make_dataset(df):
    ds = Dataset.from_pandas(df[["text","label"]].reset_index(drop=True))
    return ds.map(tokenize, batched=True)

train_ds   = make_dataset(train_df)
val_ds     = make_dataset(val_df)
siswati_ds = make_dataset(siswati_df)

print(f"Datasets have been tokenized")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


✓ Loaded tokenizer: Davlan/afro-xlmr-base


Map:   0%|          | 0/528 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

Map:   0%|          | 0/293 [00:00<?, ? examples/s]

Datasets have been tokenized


In [17]:
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

f1_metric        = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1":  f1_metric.compute(
                         predictions=preds, references=labels, average="macro")["f1"],
        "precision": precision_metric.compute(
                         predictions=preds, references=labels, average="macro")["precision"],
        "recall":    recall_metric.compute(
                         predictions=preds, references=labels, average="macro")["recall"],
    }


In [18]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)


print(" AfroXLMR on isiZulu + isiXhosa Fine Tuning ")
print("-"*55)

ft_model = AutoModelForSequenceClassification.from_pretrained(
    AFROXLMR_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=f"{BASE_DIR}/finetuned",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    logging_steps=10,
    logging_dir=f"{BASE_DIR}/logs",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()

trainer.save_model(f"{BASE_DIR}/best_model")
tokenizer.save_pretrained(f"{BASE_DIR}/best_model")
print(f"The best model has saved to {BASE_DIR}/best_model")

 AfroXLMR on isiZulu + isiXhosa Fine Tuning 
-------------------------------------------------------


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is

Starting fine-tuning...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The best model has saved to /content/drive/My Drive/afroxlmr_detector/best_model


In [19]:

print("CROSS-LINGUAL EVALUATION: Fine-tuned model on Siswati: Cross Lingual Evaluation")
print("-"*55)

ft_results = trainer.evaluate(siswati_ds)
print("\nFine-tuned results on Siswati (cross-lingual transfer):")
for k, v in ft_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")



CROSS-LINGUAL EVALUATION: Fine-tuned model on Siswati: Cross Lingual Evaluation
-------------------------------------------------------



Fine-tuned results on Siswati (cross-lingual transfer):
  eval_loss: 0.5696
  eval_macro_f1: 0.7205
  eval_precision: 0.8704
  eval_recall: 0.7091
  eval_runtime: 1.7258
  eval_samples_per_second: 169.7800
  eval_steps_per_second: 21.4400
  epoch: 5.0000


In [20]:
print("Fine-tuned AfroXLMR on Siswati Report")
print("-"*55)

preds_output = trainer.predict(siswati_ds)
ft_preds     = np.argmax(preds_output.predictions, axis=-1)
true_labels  = siswati_df["label"].values

print("\nClassification Report:")
print(classification_report(
    true_labels, ft_preds,
    target_names=["Human", "Machine"],
    digits=4
))

cm = confusion_matrix(true_labels, ft_preds)
print("Confusion Matrix (rows=true, cols=predicted):")
print(f"                Pred:Human  Pred:Machine")
print(f"  True:Human       {cm[0,0]:>6}        {cm[0,1]:>6}")
print(f"  True:Machine     {cm[1,0]:>6}        {cm[1,1]:>6}")


Fine-tuned AfroXLMR on Siswati Report
-------------------------------------------------------



Classification Report:
              precision    recall  f1-score   support

       Human     0.7409    1.0000    0.8512       183
     Machine     1.0000    0.4182    0.5897       110

    accuracy                         0.7816       293
   macro avg     0.8704    0.7091    0.7205       293
weighted avg     0.8382    0.7816    0.7530       293

Confusion Matrix (rows=true, cols=predicted):
                Pred:Human  Pred:Machine
  True:Human          183             0
  True:Machine         64            46


In [27]:
import json

print("Comparing TF-IDF Baseline (Phase 1) vs AfroXLMR (Phase 2)")
print("On Siswati zero-shot / cross-lingual test set")
print("-"*55)

if os.path.exists(PHASE1_METRICS_JSON):
    with open(PHASE1_METRICS_JSON) as f:
        phase1_all = json.load(f)
    phase1 = phase1_all.get("siswati_zeroshot", {})
    print(f"✓ Loaded Phase 1 metrics from {PHASE1_METRICS_JSON}\n")
else:
    print(f"Phase 1 metrics not found at {PHASE1_METRICS_JSON}")
    print("  Run Phase 1 first, or update PHASE1_METRICS_JSON path.\n")
    phase1 = {}

phase2 = {
    "precision": ft_results.get("eval_precision", float("nan")),
    "recall":    ft_results.get("eval_recall",    float("nan")),
    "macro_f1":  ft_results.get("eval_macro_f1",  float("nan")),
    "mcc":       float("nan"),
}


from sklearn.metrics import matthews_corrcoef
phase2["mcc"] = round(matthews_corrcoef(true_labels, ft_preds), 4)

metrics_to_compare = ["precision", "recall", "macro_f1", "mcc"]
header = f"  {'Metric':<14} {'TF-IDF + LR':>14} {'AfroXLMR FT':>14} {'Δ Change':>12}"
print(header)
print("  " + "-" * (len(header) - 2))

for m in metrics_to_compare:
    p1_val = phase1.get(m, float("nan"))
    p2_val = phase2.get(m, float("nan"))
    delta  = p2_val - p1_val if not (np.isnan(p1_val) or np.isnan(p2_val)) else float("nan")
    sign   = "+" if delta >= 0 else ""
    delta_str = f"{sign}{delta:.4f}" if not np.isnan(delta) else "N/A"
    print(f"  {m:<14} {p1_val:>14.4f} {p2_val:>14.4f} {delta_str:>12}")

f1_delta = phase2["macro_f1"] - phase1.get("macro_f1", float("nan"))
if not np.isnan(f1_delta):
    improved = f1_delta > 0
    print(f"{'Goof' if improved else 'Bad'} AfroXLMR {'outperforms' if improved else 'underperforms'} "
          f"TF-IDF baseline on Siswati by {f1_delta:+.4f} Macro-F1")

Comparing TF-IDF Baseline (Phase 1) vs AfroXLMR (Phase 2)
On Siswati zero-shot / cross-lingual test set
-------------------------------------------------------
✓ Loaded Phase 1 metrics from /content/drive/MyDrive/afroxlmr_detector/baseline_metrics.json

  Metric            TF-IDF + LR    AfroXLMR FT     Δ Change
  ---------------------------------------------------------
  precision              0.9276         0.8704      -0.0572
  recall                 0.8591         0.7091      -0.1500
  macro_f1               0.8789         0.7205      -0.1584
  mcc                    0.7837         0.5566      -0.2271
Bad AfroXLMR underperforms TF-IDF baseline on Siswati by -0.1584 Macro-F1


In [28]:
phase2_metrics = {
    "model":             AFROXLMR_MODEL,
    "train_languages":   ["zu", "xh"],
    "test_language":     "ss",
    "siswati_crosslingual": {
        "precision": phase2["precision"],
        "recall":    phase2["recall"],
        "macro_f1":  phase2["macro_f1"],
        "mcc":       phase2["mcc"],
    }
}

out_path = f"{BASE_DIR}/phase2_metrics.json"
with open(out_path, "w") as f:
    json.dump(phase2_metrics, f, indent=2)
print(f"\n Phase 2 metrics saved to {out_path}")


 Phase 2 metrics saved to /content/drive/My Drive/afroxlmr_detector/phase2_metrics.json


In [31]:
def predict(text: str, verbose=True):
    ft_model.eval()
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH
    ).to(ft_model.device)
    with torch.no_grad():
        logits = ft_model(**inputs).logits
    probs   = torch.softmax(logits, dim=-1).squeeze()
    pred_id = probs.argmax().item()
    if verbose:
        print(f"Text       : {text[:80]}{'...' if len(text)>80 else ''}")
        print(f"Prediction : {ID2LABEL[pred_id].upper()}")
        print(f"Confidence : Human={probs[0]:.3f}  Machine={probs[1]:.3f}\n")
    return ID2LABEL[pred_id], probs.tolist()

print("\n" + "="*55)
print("Sample Siswati Prediction")
print("="*55 + "\n")
for row in siswati_df.head(8).itertuples():
    pred, _ = predict(row.text, verbose=False)
    match   = LABEL2ID[pred] == row.label
    print(f"{'Good' if match else 'Bad'} True: {ID2LABEL[row.label]:<8} | Predicted: {pred:<8} | {row.text[:60]}")


Sample Siswati Prediction

Good True: human    | Predicted: human    | Kube ngumsebenti loma tima kakhulu kuvumbulula lokuchumana l
Good True: human    | Predicted: human    | “Siyakuvisisa lokubaluleka kwesikhatsi nekugcina sikhatsi le
Good True: human    | Predicted: human    | Lesikhungo sivulelwe ku sita bantfu labatisulu teku hlukunye
Good True: human    | Predicted: human    | Labalimi laba-30 batfole ematayitela alomhlaba ku Mengameli 
Good True: human    | Predicted: human    | “Kusebentisa imali ngalo kwendlulele kufana nesi kweleti les
Good True: human    | Predicted: human    | Akhuluma nesive muva nje ngalokwentekako mayelana nendlelali
Good True: human    | Predicted: human    | I-COVID-19 ligciwane lelihlasela titfo temtimba tekuphefumul
Good True: human    | Predicted: human    | Noma ‘iriphabhlikhi’ icha zwa njengelive lapho khona emandla


# FINE TUNING CODE EXPLAINATION